In [0]:
from datetime import datetime

# -----------------------------------------------------------------------------
# CONFIG: Set the source and destination paths
# -----------------------------------------------------------------------------
# SRC: Workspace upload area (where you might have uploaded/unzipped landing files).
# Note: In Free Edition, Spark can't read file:/Workspace directly, but dbutils.fs can
# access workspace-backed paths via dbfs:/Workspace/...
SRC = "dbfs:/Workspace/Users/omarhamzic1@gmail.com/nba-datalakehouse/landing"

# DST: Unity Catalog Volume landing path (governed storage). This is where we want
# your raw JSONL landing folders to live so Spark can read them reliably.
DST = "dbfs:/Volumes/workspace/nba_files/landing"

print("SRC =", SRC)
print("DST =", DST)

# -----------------------------------------------------------------------------
# Helper functions (keep notebook readable)
# -----------------------------------------------------------------------------
def _ls(path: str):
    """List a path via dbutils.fs.ls and return a Python list."""
    return list(dbutils.fs.ls(path))

def _exists(path: str) -> bool:
    """Return True if the path is accessible, else False."""
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def _rm(path: str, recurse: bool = False) -> bool:
    """Remove a file/folder. Returns True if removed, False if failed."""
    try:
        dbutils.fs.rm(path, recurse)
        return True
    except Exception:
        return False

def _cleanup_ds_store(base: str) -> None:
    """
    Remove macOS .DS_Store files recursively inside a directory tree.
    These files commonly appear when zipping/unzipping from macOS and add clutter.
    """
    removed = 0
    stack = [base]
    while stack:
        p = stack.pop()
        try:
            for f in _ls(p):
                if f.name == ".DS_Store":
                    if _rm(f.path, False):
                        removed += 1
                elif f.path.endswith("/"):
                    stack.append(f.path.rstrip("/"))
        except Exception:
            # Some paths may not be listable due to permissions; ignore and continue.
            pass
    print("Removed .DS_Store files:", removed)

# -----------------------------------------------------------------------------
# 1) Sanity check: Source exists and is listable
# -----------------------------------------------------------------------------
# If this fails, you're pointing SRC at the wrong place, or the files are not in Workspace.
print("\n=== 1) List source ===")
if not _exists(SRC):
    raise RuntimeError(f"SRC not found or not accessible: {SRC}")
display(dbutils.fs.ls(SRC))

# -----------------------------------------------------------------------------
# 2) Sanity check: Destination volume exists and is listable
# -----------------------------------------------------------------------------
# If this fails, your UC Volume might not exist or permissions are wrong.
print("\n=== 2) List destination volume ===")
if not _exists(DST):
    raise RuntimeError(f"DST not found or not accessible: {DST}")
display(dbutils.fs.ls(DST))

# -----------------------------------------------------------------------------
# 3) Copy recursively from SRC to DST
# -----------------------------------------------------------------------------
# This preserves directory structure under landing, e.g.:
#   balldontlie/games/dt=YYYY-MM-DD/part-00000.jsonl
# This is generally safe to rerun; it will copy new folders/files as they appear.
print("\n=== 3) Copy recursively SRC -> DST ===")
print(f"Copying {SRC} --> {DST} (recursive)...")
dbutils.fs.cp(SRC, DST, recurse=True)
print("Copy complete.")

# -----------------------------------------------------------------------------
# 4) Fix a common mistake: nested data/landing prefix after unzip
# -----------------------------------------------------------------------------
# If you zipped from project root, the zip might contain "data/landing/..." paths.
# When extracted into DST, it becomes:
#   DST/data/landing/balldontlie/...
# But your canonical target is:
#   DST/balldontlie/...
#
# This block detects that nested folder and merges its contents back into DST root,
# then deletes the redundant DST/data tree.
print("\n=== 4) Fix nested data/landing prefix if present ===")
nested = f"{DST}/data/landing"
if _exists(nested):
    print("Detected nested path:", nested)
    for item in _ls(nested):
        src_path = item.path.rstrip("/")
        dst_path = f"{DST}/{item.name.rstrip('/')}"
        print("Merging", src_path, "->", dst_path)
        dbutils.fs.cp(src_path, dst_path, recurse=True)
    _rm(f"{DST}/data", recurse=True)
    print("Nested prefix fixed.")
else:
    print("No nested data/landing prefix detected.")

# -----------------------------------------------------------------------------
# 5) Cleanup: remove .DS_Store artifacts in the destination
# -----------------------------------------------------------------------------
print("\n=== 5) Cleanup .DS_Store artifacts ===")
_cleanup_ds_store(DST)

# -----------------------------------------------------------------------------
# 6) Verify destination after copy + cleanup
# -----------------------------------------------------------------------------
print("\n=== 6) Verify destination after copy ===")
display(dbutils.fs.ls(DST))

# -----------------------------------------------------------------------------
# 7) Verify expected entity paths (skip if missing)
# -----------------------------------------------------------------------------
# Some sources/entities may not exist yet (e.g., the_odds_api). We don't want to fail
# the notebook for optional sources.
print("\n=== 7) Verify expected entity paths (skip if missing) ===")
expected_paths = [
    f"{DST}/balldontlie",
    f"{DST}/balldontlie/games",
    f"{DST}/balldontlie/teams",
    f"{DST}/the_odds_api",
]

for p in expected_paths:
    if _exists(p):
        listing = _ls(p)
        print(f"OK: {p} ({len(listing)} items)")
    else:
        print(f"SKIP (not found yet): {p}")

# -----------------------------------------------------------------------------
# 8) Optional: show the latest dt partition under games (if present)
# -----------------------------------------------------------------------------
# This helps confirm that your daily ingestion/backfill landed where expected.
print("\n=== 8) Show latest dt partition under games (if present) ===")
games_path = f"{DST}/balldontlie/games"
if _exists(games_path):
    dt_dirs = sorted([x.name.rstrip("/") for x in _ls(games_path) if x.name.startswith("dt=")])
    if dt_dirs:
        latest = dt_dirs[-1]
        print("Latest dt folder:", latest)
        display(dbutils.fs.ls(f"{games_path}/{latest}"))
    else:
        print("No dt= folders found under games.")
else:
    print("Games path not found:", games_path)

# -----------------------------------------------------------------------------
# Success criteria:
# - DST contains balldontlie/ and within it games/ with dt=YYYY-MM-DD folders
# - No accidental DST/data/landing nesting remains
# - .DS_Store is removed (best-effort)
#
# Next notebook should read from:
#   dbfs:/Volumes/workspace/nba_files/landing/balldontlie/games
# and build Bronze Delta tables.
# -----------------------------------------------------------------------------
